# 01 — PyTorch baselines (FP32 / FP16)

Runs:
- FP32 (CUDA)
- FP16 (CUDA)

Optionally without reduced bit depth.

In [1]:
from pathlib import Path
import sys, os

# ---- Path setup (adjust if your repo layout differs) ----
PROJECT_ROOT = Path("..").resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [2]:
# Purpose: Establish PyTorch FP32 baseline (and optional FP16 baseline) on ImageNet val.
import os
import json
from pathlib import Path
import pandas as pd

# project imports
from config import ExperimentConfig, with_overrides
from runner import run_experiment
from utils import load_runs, flatten_runs
from plots import (
    plot_metric_vs_input_bits,
    plot_tradeoff_with_pareto,
    plot_tradeoff_scatter,
)

pd.set_option("display.max_columns", 200)

In [ ]:
# Baselines (PyTorch backend only)
base = ExperimentConfig(
    backend="pytorch",
    device="cuda",             
    batch_size=1,
    input_quant_bits=8,        
    seed=42,
    num_eval_batches=500
)


In [4]:
cfg32 = with_overrides(base, model_precision="fp32")

payload, tracker = run_experiment(cfg32, split="val", save_results_flag=True)

# quick peek
print(payload["run_id"], payload["status"], payload["results"].get("top1_acc"))

Evaluating on 500 batches...
  Batch [10/500] Top-1: 80.00% | Top-5: 100.00% | Infer: 24.83 ms/batch
  Batch [20/500] Top-1: 85.00% | Top-5: 95.00% | Infer: 13.82 ms/batch
  Batch [30/500] Top-1: 90.00% | Top-5: 96.67% | Infer: 10.34 ms/batch
  Batch [40/500] Top-1: 85.00% | Top-5: 95.00% | Infer: 8.53 ms/batch
  Batch [50/500] Top-1: 88.00% | Top-5: 96.00% | Infer: 7.35 ms/batch
  Batch [60/500] Top-1: 85.00% | Top-5: 95.00% | Infer: 6.56 ms/batch
  Batch [70/500] Top-1: 84.29% | Top-5: 95.71% | Infer: 6.01 ms/batch
  Batch [80/500] Top-1: 85.00% | Top-5: 95.00% | Infer: 5.65 ms/batch
  Batch [90/500] Top-1: 84.44% | Top-5: 95.56% | Infer: 5.47 ms/batch
  Batch [100/500] Top-1: 85.00% | Top-5: 96.00% | Infer: 5.28 ms/batch
  Batch [110/500] Top-1: 83.64% | Top-5: 95.45% | Infer: 5.10 ms/batch
  Batch [120/500] Top-1: 82.50% | Top-5: 95.83% | Infer: 4.93 ms/batch
  Batch [130/500] Top-1: 82.31% | Top-5: 96.15% | Infer: 4.80 ms/batch
  Batch [140/500] Top-1: 82.86% | Top-5: 96.43% | Inf

In [6]:
# Always load from disk (the source of truth)
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

# Filter to just this notebook’s scope (pytorch + no input quant)
df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp32"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.backend",
    "cfg.device",
    "cfg.batch_size",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["cfg.model_precision", "cfg.device"])

,run_id,cfg.backend,cfg.device,cfg.batch_size,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps,res.total_samples
0,resnet18_pytorch_fp32_in8b_cuda_bs1,pytorch,cuda,1,fp32,8,79.0,94.4,3.36433,297.236015,500


In [8]:
cfg16 = with_overrides(base, model_precision="fp16")

payload, tracker = run_experiment(cfg16, split="val", save_results_flag=True)

# quick peek
print(payload["run_id"], payload["status"], payload["results"].get("top1_acc"))

Evaluating on 500 batches...
  Batch [10/500] Top-1: 80.00% | Top-5: 100.00% | Infer: 27.72 ms/batch
  Batch [20/500] Top-1: 85.00% | Top-5: 95.00% | Infer: 15.83 ms/batch
  Batch [30/500] Top-1: 90.00% | Top-5: 96.67% | Infer: 11.97 ms/batch
  Batch [40/500] Top-1: 85.00% | Top-5: 95.00% | Infer: 9.92 ms/batch
  Batch [50/500] Top-1: 88.00% | Top-5: 96.00% | Infer: 8.53 ms/batch
  Batch [60/500] Top-1: 85.00% | Top-5: 95.00% | Infer: 7.53 ms/batch
  Batch [70/500] Top-1: 84.29% | Top-5: 95.71% | Infer: 6.85 ms/batch
  Batch [80/500] Top-1: 85.00% | Top-5: 95.00% | Infer: 6.45 ms/batch
  Batch [90/500] Top-1: 84.44% | Top-5: 95.56% | Infer: 6.03 ms/batch
  Batch [100/500] Top-1: 85.00% | Top-5: 96.00% | Infer: 5.90 ms/batch
  Batch [110/500] Top-1: 83.64% | Top-5: 95.45% | Infer: 5.60 ms/batch
  Batch [120/500] Top-1: 82.50% | Top-5: 95.83% | Infer: 5.34 ms/batch
  Batch [130/500] Top-1: 82.31% | Top-5: 96.15% | Infer: 5.07 ms/batch
  Batch [140/500] Top-1: 82.86% | Top-5: 96.43% | Inf

In [9]:
# Always load from disk (the source of truth)
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

# Filter to just this notebook’s scope (pytorch + no input quant)
df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp16"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.backend",
    "cfg.device",
    "cfg.batch_size",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["cfg.model_precision", "cfg.device"])

,run_id,cfg.backend,cfg.device,cfg.batch_size,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps,res.total_samples
0,resnet18_pytorch_fp16_in8b_cuda_bs1,pytorch,cuda,1,fp16,8,79.0,94.4,3.473543,287.890477,500


In [ ]:
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp32", "fp16"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.model_precision",
    "res.infer_ms_avg",
    "res.infer_ms_min",
    "res.infer_ms_max",
]].sort_values("cfg.model_precision")